# OceanEmbed — Subsurface Temperature Reconstruction (Corrected)
**Full pipeline**: Data ingestion → Preprocessing (Prefect) → Satellite Embedding → U-Net+BiConvGRU+Attention → MLflow/W&B → ARGO Validation → XAI → Model Save

**Domain**: North Indian Ocean | 5°N–30°N, 45°E–105°E | Daily, 0.25° | 2011–2022

**Inputs**: SST (OSTIA), SSS (SMOS/SMAP), SSH/SLA (DUACS), Winds (CMEMS), Surface Currents (MULTIOBS)

**Target**: GLORYS12 thetao at 15 standard depths (0–1000 m)

---
### ✅ Fixes applied to original notebook:
- Removed duplicate data-exploration cells
- Fixed `FutureWarning`: `ds.dims` → `ds.sizes`
- Added missing `YEARS` variable in Cell 0
- Removed broken `earthaccess` NASA download cell
- Fixed `load_and_concat` to handle files present in DATA_DIR correctly
- Added `torch.cuda.amp` mixed-precision training (saves VRAM on Colab T4)
- Added `torch.backends.cudnn.benchmark = True` for faster GPU training
- Fixed `DataLoader` memory settings for Colab
- Fixed scaler application shape bugs
- Made W&B login safe (try/except)
- Added `weights_only=True` to `torch.load` for security
- Fixed ARGO profile groupby bug
- Added `BATCH = 2` default (increase if GPU allows)

## Step 0a — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 0b — Install Dependencies

In [ ]:
%%capture
!pip install copernicusmarine xarray netCDF4 zarr scipy numpy pandas \
    prefect scikit-learn mlflow wandb \
    captum matplotlib tqdm

## Cell 0 — Credentials & Global Config

In [ ]:
import os
import torch

# ─── CREDENTIALS ──────────────────────────────────────────────────────────────
COP_USER  = "franz"            # Copernicus Marine username
COP_PASS  = "Joes@9092443143" # Copernicus Marine password
WANDB_KEY = "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"

# ─── DOMAIN ───────────────────────────────────────────────────────────────────
LON_MIN, LON_MAX = 45.0, 105.0
LAT_MIN, LAT_MAX =  5.0,  30.0

# ─── YEARS ────────────────────────────────────────────────────────────────────
YEARS = list(range(2011, 2023))  # 2011 – 2022 inclusive

# ─── TIME SPLITS ──────────────────────────────────────────────────────────────
TRAIN_START, TRAIN_END = "2011-01-01", "2019-12-31"
VAL_START,   VAL_END   = "2020-01-01", "2020-12-31"
TEST_START,  TEST_END  = "2021-01-01", "2022-12-31"

# ─── TARGET DEPTHS (metres) ───────────────────────────────────────────────────
TARGET_DEPTHS = [0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000]
N_DEPTHS = len(TARGET_DEPTHS)  # 15

# ─── GRID (padded to be divisible by 8 for 3-level U-Net) ────────────────────
PAD_LAT, PAD_LON = 104, 248

# ─── PATHS ────────────────────────────────────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive'
DATA_DIR  = os.path.join(DRIVE_DIR, 'OceanEmbed', 'data', 'raw')
PROC_DIR  = os.path.join(DRIVE_DIR, 'OceanEmbed', 'data', 'processed')
CKPT_DIR  = os.path.join(DRIVE_DIR, 'OceanEmbed', 'checkpoints')
MODEL_DIR = os.path.join(DRIVE_DIR, 'OceanEmbed', 'models')

for d in [DATA_DIR, PROC_DIR, CKPT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── GPU SETUP ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be slow on CPU!')

print(f'Training on: {device}')
print('Config OK. Directories created.')

## Cell 1 — Data Ingestion from Copernicus Marine
Downloads 6 datasets for 2011–2022 subsetted to the North Indian Ocean domain.
Uses lazy open first to inspect, then `to_netcdf` year-by-year to avoid OOM.

⚠️ **This cell downloads ~50–100 GB of data to Google Drive. It skips files already present.**
If you already downloaded data, this cell will skip existing files.

In [ ]:
import copernicusmarine
import os

DATASETS = [
    # (key, dataset_id, variables, min_depth, max_depth)
    ("glorys",    "cmems_mod_glo_phy_my_0.083deg_P1D-m",
     ["thetao"], 0.0, 1000.0),
    ("sst",       "METOFFICE-GLO-SST-L4-REP-OBS-SST",
     ["analysed_sst"], None, None),
    ("sss",       "cmems_obs-mob_glo_phy-sss_my_multi_P1D",
     ["sos"], None, None),
    ("ssh",       "c3s_obs-sl_glo_phy-ssh_my_twosat-l4-duacs-0.25deg_P1D",
     ["sla", "ugos", "vgos"], None, None),
    ("winds",     "cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H",
     ["eastward_wind", "northward_wind"], None, None),
    ("currents",  "cmems_obs-mob_glo_phy-cur_my_0.25deg_P1D-m",
     ["uo", "vo"], None, None),
]

print('Starting downloads (skipping already-present files)...')

for key, dataset_id, variables, min_depth, max_depth in DATASETS:
    print(f'\n[{key.upper()}]')
    for year in YEARS:
        out_path = os.path.join(DATA_DIR, f'{key}_{year}.nc')
        if os.path.exists(out_path):
            print(f'  SKIP {key} {year} (already exists)')
            continue
        try:
            kwargs = dict(
                dataset_id=dataset_id,
                variables=variables,
                minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
                minimum_latitude=LAT_MIN,  maximum_latitude=LAT_MAX,
                start_datetime=f'{year}-01-01T00:00:00',
                end_datetime=f'{year}-12-31T23:59:59',
                output_filename=f'{key}_{year}.nc',
                output_directory=DATA_DIR,
                username=COP_USER, password=COP_PASS,
                force_download=False,
            )
            if min_depth is not None:
                kwargs['minimum_depth'] = min_depth
                kwargs['maximum_depth'] = max_depth
            copernicusmarine.subset(**kwargs)
            print(f'  OK  {key} {year} -> {out_path}')
        except Exception as e:
            print(f'  ERR {key} {year}: {e}')

print('\nDownload phase complete.')

## Cell 1b — Quick Data Inspection (Optional)

In [ ]:
import glob
import os
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

def inspect_and_explore_datasets(data_dir):
    nc_files = sorted(glob.glob(os.path.join(data_dir, '*.nc')))

    if not nc_files:
        print(f'No NetCDF (.nc) files found in {data_dir}')
        return

    print(f'Found {len(nc_files)} NetCDF file(s):\n')
    for f in nc_files:
        print(f'  - {os.path.basename(f)}')
    print('=' * 70)

    for file_path in nc_files:
        filename = os.path.basename(file_path)
        print(f'\n===== File: {filename} =====')
        try:
            ds = xr.open_dataset(file_path)

            # FIX: Use ds.sizes instead of ds.dims (avoids FutureWarning)
            print('\n--- DIMENSIONS ---')
            for dim, size in ds.sizes.items():
                print(f'  * {dim}: {size}')

            print('\n--- DATA VARIABLES & SHAPES ---')
            for var in ds.data_vars:
                var_da = ds[var]
                unit      = var_da.attrs.get('units', 'no unit')
                long_name = var_da.attrs.get('long_name', var)
                print(f'  * {var}: shape={var_da.shape}, dtype={var_da.dtype}')
                print(f'    └─ {long_name} [{unit}]')

            print('\n--- COORDINATE BOUNDS ---')
            for coord in ds.coords:
                coord_data = ds[coord].values
                if coord_data.dtype.kind in ['f', 'i']:
                    print(f'  * {coord}: Min={coord_data.min():.3f}, Max={coord_data.max():.3f}')
                elif coord_data.dtype.kind == 'M':  # datetime
                    print(f'  * {coord}: {coord_data.min()} → {coord_data.max()}')

            print('\n--- MISSING VALUES ---')
            for var in list(ds.data_vars)[:3]:  # limit to first 3 vars
                null_count  = int(ds[var].isnull().sum().values)
                total_count = ds[var].size
                pct = (null_count / total_count) * 100
                print(f'  * {var}: {null_count}/{total_count} ({pct:.2f}% NaN)')

            ds.close()
        except Exception as e:
            print(f'  ERROR: {e}')

inspect_and_explore_datasets(DATA_DIR)

## Cell 2 — Preprocessing & Harmonisation (Prefect Flow)
- Regrid everything to 0.25° target grid (101×241)
- Daily-average winds (hourly → daily)
- Fill gaps with linear interpolation
- Interpolate GLORYS thetao to the 15 standard depths
- Compute daily climatology anomaly ΔT = T − T_clim (from training years only)
- Pad spatial dimensions to 104×248
- Save combined Zarr store

In [ ]:
import glob
import os
import numpy as np
import xarray as xr
from scipy.interpolate import RegularGridInterpolator
from prefect import flow, task

# ─── Target grid ──────────────────────────────────────────────────────────────
TARGET_LAT = np.arange(LAT_MIN, LAT_MAX + 0.25, 0.25)   # 5.0 .. 30.0 → 101 pts
TARGET_LON = np.arange(LON_MIN, LON_MAX + 0.25, 0.25)   # 45.0 .. 105.0 → 241 pts

@task(name="load_concat")
def load_and_concat(key: str, years: list) -> xr.Dataset:
    """Open all annual files for a key and concatenate along time."""
    paths = sorted([
        os.path.join(DATA_DIR, f'{key}_{yr}.nc')
        for yr in years
        if os.path.exists(os.path.join(DATA_DIR, f'{key}_{yr}.nc'))
    ])
    if not paths:
        raise FileNotFoundError(f'No files found for key={key!r} in {DATA_DIR}')
    ds = xr.open_mfdataset(paths, combine='by_coords', engine='netcdf4')
    # Standardise coordinate names to lat/lon/time
    rename = {}
    for dim in list(ds.dims):  # FIX: wrap in list() to avoid dict-change warnings
        if 'latitude'  in dim.lower(): rename[dim] = 'lat'
        if 'longitude' in dim.lower(): rename[dim] = 'lon'
    if rename:
        ds = ds.rename(rename)
    return ds

@task(name="regrid_025")
def regrid_to_025(ds: xr.Dataset, varnames: list) -> xr.Dataset:
    """Bilinear regrid to the 0.25° target grid using scipy."""
    out_vars = {}
    src_lat = ds['lat'].values.astype(float)
    src_lon = ds['lon'].values.astype(float)
    times   = ds['time'].values

    # Ensure src_lat is ascending (required by RegularGridInterpolator)
    lat_flip = src_lat[0] > src_lat[-1]
    if lat_flip:
        src_lat = src_lat[::-1]

    for vn in varnames:
        da = ds[vn]
        if 'depth' in da.dims:
            layers = []
            for dep in da['depth'].values:
                slices = []
                for t in range(len(times)):
                    arr = da.isel(time=t).sel(depth=dep, method='nearest').values
                    if lat_flip:
                        arr = arr[::-1]
                    interp = RegularGridInterpolator(
                        (src_lat, src_lon), arr,
                        method='linear', bounds_error=False, fill_value=np.nan)
                    lo_g, la_g = np.meshgrid(TARGET_LON, TARGET_LAT)
                    pts = np.column_stack([la_g.ravel(), lo_g.ravel()])
                    slices.append(interp(pts).reshape(len(TARGET_LAT), len(TARGET_LON)))
                layers.append(np.stack(slices, axis=0))
            out_vars[vn] = xr.DataArray(
                np.stack(layers, axis=1),
                dims=['time', 'depth', 'lat', 'lon'],
                coords={'time': times, 'depth': da['depth'].values,
                        'lat': TARGET_LAT, 'lon': TARGET_LON})
        else:
            slices = []
            for t in range(len(times)):
                arr = da.isel(time=t).values.squeeze()
                if lat_flip:
                    arr = arr[::-1]
                interp = RegularGridInterpolator(
                    (src_lat, src_lon), arr,
                    method='linear', bounds_error=False, fill_value=np.nan)
                lo_g, la_g = np.meshgrid(TARGET_LON, TARGET_LAT)
                pts = np.column_stack([la_g.ravel(), lo_g.ravel()])
                slices.append(interp(pts).reshape(len(TARGET_LAT), len(TARGET_LON)))
            out_vars[vn] = xr.DataArray(
                np.stack(slices, axis=0),
                dims=['time', 'lat', 'lon'],
                coords={'time': times, 'lat': TARGET_LAT, 'lon': TARGET_LON})
    return xr.Dataset(out_vars)

@task(name="daily_average")
def daily_average(ds: xr.Dataset) -> xr.Dataset:
    """Resample sub-daily data to daily mean."""
    return ds.resample(time='1D').mean()

@task(name="fill_gaps")
def fill_gaps(ds: xr.Dataset) -> xr.Dataset:
    """Linear interpolation in time to fill NaN gaps."""
    return ds.interpolate_na(dim='time', method='linear', fill_value='extrapolate')

@task(name="interp_depths")
def interp_to_standard_depths(ds: xr.Dataset, varname: str = 'thetao') -> xr.Dataset:
    """Interpolate GLORYS thetao to the 15 standard depth levels."""
    out = ds[varname].interp(
        depth=xr.DataArray(TARGET_DEPTHS, dims='depth'),
        method='linear'
    )
    return xr.Dataset({varname: out})

@task(name="compute_anomaly")
def compute_anomaly(ds_full: xr.Dataset, ds_train: xr.Dataset, varname: str) -> xr.Dataset:
    """Compute ΔT = T − daily_climatology. Climatology from training years only."""
    clim = ds_train[varname].groupby('time.dayofyear').mean('time')
    anom = ds_full[varname].groupby('time.dayofyear') - clim
    return xr.Dataset({f'{varname}_anom': anom, f'{varname}_clim': clim})

@task(name="pad_spatial")
def pad_spatial(da: xr.DataArray) -> xr.DataArray:
    """Reflect-pad lat from 101→104, lon from 241→248 for divisibility by 8."""
    da_np   = da.values
    lat_ax  = list(da.dims).index('lat')
    lon_ax  = list(da.dims).index('lon')
    pad_cfg = [(0, 0)] * da_np.ndim
    pad_cfg[lat_ax] = (0, PAD_LAT - da.sizes['lat'])
    pad_cfg[lon_ax] = (0, PAD_LON - da.sizes['lon'])
    da_padded = np.pad(da_np, pad_cfg, mode='reflect')
    new_lat   = np.linspace(TARGET_LAT[0], TARGET_LAT[0] + (PAD_LAT - 1) * 0.25, PAD_LAT)
    new_lon   = np.linspace(TARGET_LON[0], TARGET_LON[0] + (PAD_LON - 1) * 0.25, PAD_LON)
    new_coords = dict(da.coords)
    new_coords['lat'] = new_lat
    new_coords['lon'] = new_lon
    return xr.DataArray(da_padded, dims=da.dims, coords=new_coords)

@flow(name="OceanEmbed_Preprocessing")
def preprocess_flow():
    all_years   = YEARS
    train_years = [y for y in all_years if 2011 <= y <= 2019]

    print('[1/6] Loading and concatenating raw files...')
    ds_glorys   = load_and_concat('glorys',   all_years)
    ds_sst      = load_and_concat('sst',      all_years)
    ds_sss      = load_and_concat('sss',      all_years)
    ds_ssh      = load_and_concat('ssh',      all_years)
    ds_winds    = load_and_concat('winds',    all_years)
    ds_currents = load_and_concat('currents', all_years)

    print('[2/6] Regridding to 0.25°...')
    ds_glorys_r   = regrid_to_025(ds_glorys,   ['thetao'])
    ds_sst_r      = regrid_to_025(ds_sst,      ['analysed_sst'])
    ds_sss_r      = regrid_to_025(ds_sss,      ['sos'])
    ds_ssh_r      = regrid_to_025(ds_ssh,      ['sla', 'ugos', 'vgos'])
    ds_winds_r    = regrid_to_025(ds_winds,    ['eastward_wind', 'northward_wind'])
    ds_currents_r = regrid_to_025(ds_currents, ['uo', 'vo'])

    print('[3/6] Daily averaging winds...')
    ds_winds_d = daily_average(ds_winds_r)

    print('[4/6] Filling gaps...')
    ds_sst_f      = fill_gaps(ds_sst_r)
    ds_sss_f      = fill_gaps(ds_sss_r)
    ds_ssh_f      = fill_gaps(ds_ssh_r)
    ds_winds_f    = fill_gaps(ds_winds_d)
    ds_currents_f = fill_gaps(ds_currents_r)
    ds_glorys_f   = fill_gaps(ds_glorys_r)

    print('[5/6] Interpolating GLORYS to standard depths & computing anomaly...')
    ds_glorys_15d   = interp_to_standard_depths(ds_glorys_f, 'thetao')
    ds_glorys_train = ds_glorys_15d.sel(time=slice(str(min(train_years)), str(max(train_years))))
    ds_anom         = compute_anomaly(ds_glorys_15d, ds_glorys_train, 'thetao')

    print('[6/6] Merging, padding, saving to Zarr...')
    common_time = ds_glorys_15d['time']
    surf = xr.merge([
        ds_sst_f.sel(time=common_time, method='nearest').rename({'analysed_sst': 'sst'}),
        ds_sss_f.sel(time=common_time, method='nearest').rename({'sos': 'sss'}),
        ds_ssh_f.sel(time=common_time, method='nearest'),
        ds_winds_f.sel(time=common_time, method='nearest').rename(
            {'eastward_wind': 'wind_u', 'northward_wind': 'wind_v'}),
        ds_currents_f.sel(time=common_time, method='nearest').rename(
            {'uo': 'cur_u', 'vo': 'cur_v'}),
    ])

    surf_vars = ['sst', 'sss', 'sla', 'ugos', 'vgos', 'wind_u', 'wind_v', 'cur_u', 'cur_v']
    for vn in surf_vars:
        surf[vn] = pad_spatial(surf[vn])
    ds_anom['thetao_anom'] = pad_spatial(ds_anom['thetao_anom'])

    surf.to_zarr(os.path.join(PROC_DIR, 'surface_inputs.zarr'),    mode='w')
    ds_anom.to_zarr(os.path.join(PROC_DIR, 'target_anomaly.zarr'), mode='w')
    ds_glorys_15d.to_zarr(os.path.join(PROC_DIR, 'target_absolute.zarr'), mode='w')
    print('Preprocessing complete. Zarr stores saved.')
    return surf, ds_anom

surf_ds, anom_ds = preprocess_flow()

## Cell 3 — Satellite Embedding Engine
A CNN autoencoder compresses 9 surface channels → 64-channel latent representation per time step.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    """Conv2d → BN → ReLU × 2."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class SatelliteEmbeddingEncoder(nn.Module):
    """
    CNN encoder: (B, 9, 104, 248) → latent (B, 64, 13, 31).
    Three downsampling levels + 1×1 conv bottleneck.
    """
    def __init__(self, in_ch=9, latent_ch=64):
        super().__init__()
        self.enc1      = ConvBlock(in_ch, 32)           # (B,32,104,248)
        self.pool1     = nn.MaxPool2d(2)                 # (B,32, 52,124)
        self.enc2      = ConvBlock(32, 64)               # (B,64, 52,124)
        self.pool2     = nn.MaxPool2d(2)                 # (B,64, 26, 62)
        self.enc3      = ConvBlock(64, 128)              # (B,128,26, 62)
        self.pool3     = nn.MaxPool2d(2)                 # (B,128,13, 31)
        self.bottleneck = nn.Conv2d(128, latent_ch, 1)  # (B,64, 13, 31)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool1(s1))
        s3 = self.enc3(self.pool2(s2))
        z  = self.bottleneck(self.pool3(s3))
        return z, s1, s2, s3

# Quick shape check
print('Embedding encoder defined.')
_dummy = torch.zeros(2, 9, 104, 248)
_enc   = SatelliteEmbeddingEncoder()
_z, _s1, _s2, _s3 = _enc(_dummy)
print(f'  Input : {tuple(_dummy.shape)}')
print(f'  Latent: {tuple(_z.shape)}')
print(f'  Skip1 : {tuple(_s1.shape)}  Skip2: {tuple(_s2.shape)}  Skip3: {tuple(_s3.shape)}')

## Cell 4 — Full Model: U-Net + Bidirectional ConvGRU + Coordinate Attention

In [ ]:
class CoordAttention(nn.Module):
    """Channel + spatial coordinate attention on skip connections."""
    def __init__(self, ch):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mid = max(8, ch // 4)
        self.mlp    = nn.Sequential(
            nn.Conv2d(ch, mid, 1, bias=False), nn.ReLU(inplace=True))
        self.conv_h = nn.Conv2d(mid, ch, 1, bias=False)
        self.conv_w = nn.Conv2d(mid, ch, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        h   = self.pool_h(x)               # (B,C,H,1)
        w   = self.pool_w(x)               # (B,C,1,W)
        w_t = w.permute(0, 1, 3, 2)        # (B,C,W,1)
        hw  = torch.cat([h, w_t], dim=2)   # (B,C,H+W,1)
        hw  = self.mlp(hw)
        ah, aw = hw.split([H, W], dim=2)
        aw  = aw.permute(0, 1, 3, 2)
        return x * self.conv_h(ah).sigmoid() * self.conv_w(aw).sigmoid()

class BiConvGRUCell(nn.Module):
    """Single BiConvGRU cell operating on (B, C, H, W)."""
    def __init__(self, in_ch, h_ch, kernel=3):
        super().__init__()
        pad = kernel // 2
        self.h_ch = h_ch
        self.reset_gate  = nn.Conv2d(in_ch + h_ch, h_ch, kernel, padding=pad)
        self.update_gate = nn.Conv2d(in_ch + h_ch, h_ch, kernel, padding=pad)
        self.out_gate    = nn.Conv2d(in_ch + h_ch, h_ch, kernel, padding=pad)

    def forward(self, x, h):
        if h is None:
            h = torch.zeros(x.size(0), self.h_ch, x.size(2), x.size(3),
                            device=x.device, dtype=x.dtype)
        xh  = torch.cat([x, h], dim=1)
        r   = torch.sigmoid(self.reset_gate(xh))
        u   = torch.sigmoid(self.update_gate(xh))
        xrh = torch.cat([x, r * h], dim=1)
        n   = torch.tanh(self.out_gate(xrh))
        return (1 - u) * h + u * n

class BiConvGRU(nn.Module):
    """Bidirectional ConvGRU over a sequence of latent frames."""
    def __init__(self, in_ch, h_ch):
        super().__init__()
        self.fwd   = BiConvGRUCell(in_ch, h_ch)
        self.bwd   = BiConvGRUCell(in_ch, h_ch)
        self.merge = nn.Conv2d(h_ch * 2, in_ch, 1)

    def forward(self, seq):  # seq: list of T tensors (B, C, H, W)
        T = len(seq)
        fwd_h, bwd_h = None, None
        fwd_outs, bwd_outs = [], []
        for t in range(T):
            fwd_h = self.fwd(seq[t], fwd_h)
            fwd_outs.append(fwd_h)
        for t in range(T - 1, -1, -1):
            bwd_h = self.bwd(seq[t], bwd_h)
            bwd_outs.insert(0, bwd_h)
        merged = [self.merge(torch.cat([f, b], dim=1))
                  for f, b in zip(fwd_outs, bwd_outs)]
        return merged

class UpBlock(nn.Module):
    """Upsample + ConvBlock + skip concat."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.attn = CoordAttention(skip_ch)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x    = self.up(x)
        skip = self.attn(skip)
        x    = torch.cat([x, skip], dim=1)
        return self.conv(x)

class OceanEmbedModel(nn.Module):
    """
    Full model:
      - SatelliteEmbeddingEncoder per frame
      - BiConvGRU over T-frame window at bottleneck
      - U-Net decoder with CoordAttention on skips
      - Output: (B, N_DEPTHS, 104, 248)
    """
    def __init__(self, in_ch=9, latent_ch=64, h_ch=64, out_depths=15):
        super().__init__()
        self.encoder = SatelliteEmbeddingEncoder(in_ch, latent_ch)
        self.bigru   = BiConvGRU(latent_ch, h_ch)
        self.up3     = UpBlock(latent_ch, 128, 64)
        self.up2     = UpBlock(64,        64,  32)
        self.up1     = UpBlock(32,        32,  32)
        self.head    = nn.Conv2d(32, out_depths, 1)

    def forward(self, x_seq):  # x_seq: (B, T, C, H, W)
        B, T, C, H, W = x_seq.shape
        zs, s1s, s2s, s3s = [], [], [], []
        for t in range(T):
            z, s1, s2, s3 = self.encoder(x_seq[:, t])
            zs.append(z); s1s.append(s1); s2s.append(s2); s3s.append(s3)

        gru_outs = self.bigru(zs)   # list of T
        mid      = T // 2           # centre frame
        z_t, s1_t, s2_t, s3_t = gru_outs[mid], s1s[mid], s2s[mid], s3s[mid]

        x = self.up3(z_t,  s3_t)
        x = self.up2(x,    s2_t)
        x = self.up1(x,    s1_t)
        return self.head(x)  # (B, 15, 104, 248)

# Shape verification
T_WIN  = 5
N_SURF = 9
model_test   = OceanEmbedModel(in_ch=N_SURF, out_depths=N_DEPTHS)
dummy_seq    = torch.zeros(2, T_WIN, N_SURF, PAD_LAT, PAD_LON)
out_test     = model_test(dummy_seq)
total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Model output shape : {tuple(out_test.shape)}  (expect (2, 15, 104, 248))')
print(f'Trainable parameters: {total_params:,}')
del model_test, dummy_seq, out_test

## Cell 5 — Dataset Class & DataLoaders
Strict temporal split (no shuffle). Window of T=5 consecutive days as input.

In [ ]:
import numpy as np
import xarray as xr
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

SURF_VARS = ['sst', 'sss', 'sla', 'ugos', 'vgos', 'wind_u', 'wind_v', 'cur_u', 'cur_v']
T_WIN     = 5   # days of context

class OceanWindowDataset(Dataset):
    """
    Returns (X, Y) where:
      X: (T_WIN, N_SURF, PAD_LAT, PAD_LON)  — surface window centred on day t
      Y: (N_DEPTHS, PAD_LAT, PAD_LON)       — subsurface anomaly on day t
    """
    def __init__(self, surf: np.ndarray, target: np.ndarray):
        self.surf   = torch.tensor(surf,   dtype=torch.float32)
        self.target = torch.tensor(target, dtype=torch.float32)
        half = T_WIN // 2
        self.indices = list(range(half, len(surf) - half))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        t    = self.indices[idx]
        half = T_WIN // 2
        x    = self.surf[t - half: t + half + 1]   # (T_WIN, N_SURF, H, W)
        y    = self.target[t]                        # (N_DEPTHS, H, W)
        return x, y

def build_arrays(surf_ds, anom_ds, time_slice):
    """Pull numpy arrays for a time slice. Returns surf (T, 9, H, W) and target (T, 15, H, W)."""
    s = surf_ds.sel(time=time_slice)
    a = anom_ds.sel(time=time_slice)
    surf_arr   = np.stack([s[v].values for v in SURF_VARS], axis=1)  # (T, 9, H, W)
    target_arr = a['thetao_anom'].values  # (T, depth, H, W)
    # FIX: Ensure depth is dim 1
    if target_arr.ndim == 4 and target_arr.shape[1] != N_DEPTHS:
        target_arr = target_arr.transpose(0, 3, 1, 2)
    return surf_arr.astype(np.float32), target_arr.astype(np.float32)

# Load processed Zarr stores
surf_z = xr.open_zarr(os.path.join(PROC_DIR, 'surface_inputs.zarr'))
anom_z = xr.open_zarr(os.path.join(PROC_DIR, 'target_anomaly.zarr'))

X_tr, Y_tr = build_arrays(surf_z, anom_z, slice(TRAIN_START, TRAIN_END))
X_va, Y_va = build_arrays(surf_z, anom_z, slice(VAL_START,   VAL_END))
X_te, Y_te = build_arrays(surf_z, anom_z, slice(TEST_START,  TEST_END))

print(f'Train: X={X_tr.shape}  Y={Y_tr.shape}')
print(f'Val  : X={X_va.shape}  Y={Y_va.shape}')
print(f'Test : X={X_te.shape}  Y={Y_te.shape}')

# Per-channel normalisation (fit on training data only)
T_tr, C, H, W = X_tr.shape
scaler_X = StandardScaler()
scaler_X.fit(X_tr.transpose(0, 2, 3, 1).reshape(-1, C))

D = Y_tr.shape[1]
scaler_Y = StandardScaler()
scaler_Y.fit(Y_tr.transpose(0, 2, 3, 1).reshape(-1, D))

def apply_scaler_X(arr):
    T_, C_, H_, W_ = arr.shape
    out = scaler_X.transform(arr.transpose(0, 2, 3, 1).reshape(-1, C_))
    return out.reshape(T_, H_, W_, C_).transpose(0, 3, 1, 2).astype(np.float32)

def apply_scaler_Y(arr):
    T_, D_, H_, W_ = arr.shape
    out = scaler_Y.transform(arr.transpose(0, 2, 3, 1).reshape(-1, D_))
    return out.reshape(T_, H_, W_, D_).transpose(0, 3, 1, 2).astype(np.float32)

X_tr_n = apply_scaler_X(X_tr);  X_va_n = apply_scaler_X(X_va);  X_te_n = apply_scaler_X(X_te)
Y_tr_n = apply_scaler_Y(Y_tr);  Y_va_n = apply_scaler_Y(Y_va);  Y_te_n = apply_scaler_Y(Y_te)

# FIX: BATCH=2 for safety; increase to 4–8 if GPU has enough VRAM
BATCH = 2
# FIX: num_workers=2 on Colab, pin_memory=True for faster GPU transfers
dl_kwargs = dict(batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
tr_loader = DataLoader(OceanWindowDataset(X_tr_n, Y_tr_n), **dl_kwargs)
va_loader = DataLoader(OceanWindowDataset(X_va_n, Y_va_n), **dl_kwargs)
te_loader = DataLoader(OceanWindowDataset(X_te_n, Y_te_n), **dl_kwargs)
print(f'Train batches: {len(tr_loader)}  Val: {len(va_loader)}  Test: {len(te_loader)}')

## Cell 6 — MLflow + W&B + Grid Search + Early Stopping + AMP Training

In [ ]:
import itertools
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import numpy as np
import mlflow
import mlflow.pytorch
import wandb

# ─── W&B login (safe — won't crash if key is invalid) ─────────────────────────
try:
    wandb.login(key=WANDB_KEY)
    USE_WANDB = True
    print('W&B logged in.')
except Exception as e:
    USE_WANDB = False
    print(f'W&B login failed ({e}) — continuing without W&B logging.')

# ─── MLflow experiment ────────────────────────────────────────────────────────
mlflow.set_tracking_uri(os.path.join(MODEL_DIR, 'mlruns'))
mlflow.set_experiment('OceanEmbed_GridSearch')

# ─── Loss: masked per-depth normalised MSE ───────────────────────────────────
class MaskedDepthNormMSE(nn.Module):
    """MSE normalised per depth level, masked where target is NaN."""
    def forward(self, pred, target):
        mask   = ~torch.isnan(target)
        pred   = pred * mask
        target = torch.nan_to_num(target, nan=0.0)
        per_depth_var = target.var(dim=[0, 2, 3], keepdim=True).clamp(min=1e-6)
        sq_err = ((pred - target) ** 2) / per_depth_var
        return sq_err[mask].mean()

criterion = MaskedDepthNormMSE()

# ─── Grid search space ────────────────────────────────────────────────────────
PARAM_GRID = {
    'lr':           [1e-3, 3e-4],
    'latent_ch':    [64],
    'h_ch':         [64, 32],
    'weight_decay': [1e-4],
}
MAX_EPOCHS = 80
PATIENCE   = 12
USE_AMP    = torch.cuda.is_available()  # Mixed precision only on GPU

print(f'Training on : {device}')
print(f'AMP enabled : {USE_AMP}')

best_global_val = math.inf
best_run_name   = None

def train_one_config(cfg, run_idx, total_runs):
    global best_global_val, best_run_name
    run_name   = f'run{run_idx:02d}_lr{cfg["lr"]}_hch{cfg["h_ch"]}'
    print(f'\n[{run_idx}/{total_runs}] {run_name}  cfg={cfg}')

    model = OceanEmbedModel(
        in_ch=N_SURF, latent_ch=cfg['latent_ch'],
        h_ch=cfg['h_ch'], out_depths=N_DEPTHS
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    scaler    = GradScaler(enabled=USE_AMP)   # AMP gradient scaler

    if USE_WANDB:
        wandb.init(project='oceanembed', name=run_name, config=cfg, reinit=True)

    best_val   = math.inf
    patience_c = 0
    ckpt_path  = os.path.join(CKPT_DIR, f'{run_name}_best.pth')

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(cfg)

        for epoch in range(1, MAX_EPOCHS + 1):
            # ── Train ──
            model.train()
            tr_loss = 0.0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)  # FIX: faster than zero_grad()
                with autocast(enabled=USE_AMP):         # FIX: mixed precision
                    pred = model(xb)
                    loss = criterion(pred, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                tr_loss += loss.item()
            tr_loss /= len(tr_loader)

            # ── Validate ──
            model.eval()
            va_loss = 0.0
            with torch.no_grad():
                for xb, yb in va_loader:
                    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                    with autocast(enabled=USE_AMP):
                        pred = model(xb)
                    va_loss += criterion(pred, yb).item()
            va_loss /= len(va_loader)
            scheduler.step()

            # ── Log ──
            mlflow.log_metrics({'train_loss': tr_loss, 'val_loss': va_loss}, step=epoch)
            if USE_WANDB:
                wandb.log({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': va_loss})

            if epoch % 10 == 0 or epoch == 1:
                print(f'  Ep {epoch:03d} | tr={tr_loss:.5f} | va={va_loss:.5f}')

            # ── Early stopping & checkpoint ──
            if va_loss < best_val:
                best_val   = va_loss
                patience_c = 0
                torch.save({
                    'epoch': epoch, 'model_state': model.state_dict(),
                    'opt_state': optimizer.state_dict(),
                    'val_loss': best_val, 'config': cfg,
                    'scaler_X_mean':  scaler_X.mean_,  'scaler_X_scale': scaler_X.scale_,
                    'scaler_Y_mean':  scaler_Y.mean_,  'scaler_Y_scale': scaler_Y.scale_,
                }, ckpt_path)
            else:
                patience_c += 1
                if patience_c >= PATIENCE:
                    print(f'  Early stop at epoch {epoch}')
                    break

        # ── Test evaluation in physical units (°C) ──
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)  # FIX: weights_only
        model.load_state_dict(ckpt['model_state'])
        model.eval()

        all_pred, all_true = [], []
        with torch.no_grad():
            for xb, yb in te_loader:
                with autocast(enabled=USE_AMP):
                    all_pred.append(model(xb.to(device)).cpu().float().numpy())
                all_true.append(yb.numpy())

        pred_sc = np.concatenate(all_pred, 0)
        true_sc = np.concatenate(all_true, 0)
        N, D, H, W = pred_sc.shape

        # FIX: correct inverse scaling
        def inv_scale(arr):
            N_, D_, H_, W_ = arr.shape
            return scaler_Y.inverse_transform(
                arr.transpose(0, 2, 3, 1).reshape(-1, D_)
            ).reshape(N_, H_, W_, D_).transpose(0, 3, 1, 2)

        pred_C = inv_scale(pred_sc)
        true_C = inv_scale(true_sc)

        mse  = float(np.nanmean((pred_C - true_C) ** 2))
        rmse = float(np.sqrt(mse))
        per_depth_rmse = {f'rmse_{TARGET_DEPTHS[d]}m': float(np.sqrt(np.nanmean((pred_C[:, d] - true_C[:, d]) ** 2)))
                          for d in range(D)}

        print(f'  TEST MSE={mse:.4f} °C²  RMSE={rmse:.4f} °C')
        mlflow.log_metrics({'test_mse_degC2': mse, 'test_rmse_degC': rmse, **per_depth_rmse})
        if USE_WANDB:
            wandb.log({'test_mse': mse, 'test_rmse': rmse, **per_depth_rmse})
        mlflow.log_artifact(ckpt_path)

        if best_val < best_global_val:
            best_global_val = best_val
            best_run_name   = run_name
            mlflow.pytorch.log_model(model, 'best_model')

    if USE_WANDB:
        wandb.finish()
    return best_val

# ─── Run grid search ──────────────────────────────────────────────────────────
keys, vals  = zip(*PARAM_GRID.items())
all_configs = [dict(zip(keys, v)) for v in itertools.product(*vals)]
print(f'Grid search: {len(all_configs)} configurations')

for i, cfg in enumerate(all_configs, 1):
    train_one_config(cfg, i, len(all_configs))

print(f'\nBest run: {best_run_name}  val_loss={best_global_val:.6f}')

## Cell 7 — ARGO In-Situ Validation
Compares model predictions at ARGO float locations against real profiles.

In [ ]:
import copernicusmarine
import pandas as pd
import numpy as np
import xarray as xr
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

# ─── 1. Load best checkpoint ──────────────────────────────────────────────────
best_ckpt_path = os.path.join(CKPT_DIR, f'{best_run_name}_best.pth')
ckpt = torch.load(best_ckpt_path, map_location='cpu', weights_only=True)  # FIX
cfg  = ckpt['config']

best_model = OceanEmbedModel(
    in_ch=N_SURF, latent_ch=cfg['latent_ch'],
    h_ch=cfg['h_ch'], out_depths=N_DEPTHS
)
best_model.load_state_dict(ckpt['model_state'])
best_model.eval()
print('Best model loaded.')

# Restore scalers
sc_X = StandardScaler()
sc_X.mean_  = ckpt['scaler_X_mean']
sc_X.scale_ = ckpt['scaler_X_scale']
sc_Y = StandardScaler()
sc_Y.mean_  = ckpt['scaler_Y_mean']
sc_Y.scale_ = ckpt['scaler_Y_scale']

# ─── 2. Download ARGO profiles ────────────────────────────────────────────────
print('Fetching ARGO profiles from CMEMS...')
argo_ok = False
try:
    argo_ds = copernicusmarine.open_dataset(
        dataset_id='cmems_obs-ins_glo_phy-ts_my_na_irr',
        variables=['TEMP', 'PRES', 'LATITUDE', 'LONGITUDE', 'TIME'],
        minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
        minimum_latitude=LAT_MIN,  maximum_latitude=LAT_MAX,
        start_datetime=TEST_START, end_datetime=TEST_END,
        username=COP_USER, password=COP_PASS,
    )
    print(f'ARGO profiles loaded: {dict(argo_ds.sizes)}')
    argo_ok = True
except Exception as e:
    print(f'ARGO fetch error: {e}')
    print('Falling back to GLORYS held-out profiles as proxy validation.')

# ─── 3. Point-wise comparison ─────────────────────────────────────────────────
def predict_at_location(lat_pt, lon_pt, date_str):
    """Get model prediction profile at a lat/lon point for a given date."""
    li = int(np.argmin(np.abs(TARGET_LAT - lat_pt)))
    lj = int(np.argmin(np.abs(TARGET_LON - lon_pt)))

    t_arr = surf_z['time'].values.astype('datetime64[D]')
    t_idx = np.where(t_arr == np.datetime64(date_str, 'D'))[0]
    if len(t_idx) == 0:
        return None
    t    = int(t_idx[0])
    half = T_WIN // 2
    if t < half or t >= len(surf_z['time']) - half:
        return None

    window = np.stack(
        [surf_z[v].isel(time=slice(t - half, t + half + 1)).values
         for v in SURF_VARS], axis=1)  # (T_WIN, N_SURF, H, W)

    T_, C_, H_, W_ = window.shape
    window_n = sc_X.transform(window.transpose(0, 2, 3, 1).reshape(-1, C_))
    window_n = window_n.reshape(T_, H_, W_, C_).transpose(0, 3, 1, 2).astype(np.float32)

    x_tensor = torch.tensor(window_n).unsqueeze(0)
    with torch.no_grad():
        pred_scaled = best_model(x_tensor).squeeze(0).numpy()

    D_, H_, W_ = pred_scaled.shape
    pred_C = sc_Y.inverse_transform(pred_scaled.transpose(1, 2, 0).reshape(-1, D_))
    pred_C = pred_C.reshape(H_, W_, D_).transpose(2, 0, 1)
    return pred_C[:, li, lj]

# ─── 4. Compute validation statistics ────────────────────────────────────────
print('\nRunning ARGO validation...')
profile_errors = []

if argo_ok:
    argo_df = argo_ds.to_dataframe().dropna(subset=['TEMP', 'PRES', 'LATITUDE', 'LONGITUDE'])
    # FIX: use reset_index instead of get_level_values for safer date extraction
    argo_df = argo_df.reset_index()
    argo_df['date'] = pd.to_datetime(argo_df['TIME']).dt.strftime('%Y-%m-%d')
    sample = argo_df.groupby('date').first().reset_index().sample(min(500, len(argo_df)))

    for _, row in sample.iterrows():
        pred_prof = predict_at_location(row['LATITUDE'], row['LONGITUDE'], row['date'])
        if pred_prof is None:
            continue
        argo_interp = interp1d(row['PRES'], row['TEMP'], bounds_error=False, fill_value=np.nan)
        argo_prof   = argo_interp(np.array(TARGET_DEPTHS, dtype=float))
        for d_idx, depth in enumerate(TARGET_DEPTHS):
            if not np.isnan(argo_prof[d_idx]):
                profile_errors.append((depth, pred_prof[d_idx] - argo_prof[d_idx]))
else:
    # Fallback: compare against GLORYS test profiles
    for xb, yb in te_loader:
        with torch.no_grad():
            pred = best_model(xb).numpy()
        B_, D_, H_, W_ = pred.shape
        pred_C = sc_Y.inverse_transform(pred.transpose(0, 2, 3, 1).reshape(-1, D_)).reshape(B_, H_, W_, D_).transpose(0, 3, 1, 2)
        true_C = sc_Y.inverse_transform(yb.numpy().transpose(0, 2, 3, 1).reshape(-1, D_)).reshape(B_, H_, W_, D_).transpose(0, 3, 1, 2)
        for b in range(B_):
            for d_idx, depth in enumerate(TARGET_DEPTHS):
                err = float(np.nanmean(pred_C[b, d_idx] - true_C[b, d_idx]))
                profile_errors.append((depth, err))
        if len(profile_errors) > 5000:
            break

if profile_errors:
    err_df = pd.DataFrame(profile_errors, columns=['depth', 'error'])
    stats  = err_df.groupby('depth')['error'].agg(
        bias='mean', rmse=lambda e: np.sqrt((e**2).mean()), std='std')
    print('\nPer-depth validation statistics (°C):')
    print(stats.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    depths = stats.index.values
    axes[0].barh(depths, stats['bias'], xerr=stats['std'], color='steelblue', alpha=0.7)
    axes[0].axvline(0, color='k', linewidth=0.8)
    axes[0].set_xlabel('Bias (°C)');  axes[0].set_ylabel('Depth (m)')
    axes[0].set_title('Bias vs Depth'); axes[0].invert_yaxis()

    axes[1].plot(stats['rmse'], depths, marker='o', color='firebrick')
    axes[1].set_xlabel('RMSE (°C)');  axes[1].set_title('RMSE vs Depth')
    axes[1].invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'argo_validation.png'), dpi=150)
    plt.show()
    print('Validation plot saved.')

## Cell 8 — Explainable AI (Captum Integrated Gradients)
Identifies which surface variables and spatial regions drive the subsurface prediction.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from captum.attr import IntegratedGradients

best_model.eval()
best_model_device = next(best_model.parameters()).device

# ─── Wrapper: predict mean temperature at 100 m (depth index 7) ──────────────
def model_scalar_100m(x_seq):
    """x_seq: (B, T, C, H, W) → scalar (B,)"""
    out = best_model(x_seq)   # (B, D, H, W)
    return out[:, 7, :, :].mean(dim=(-1, -2))

xb_xai, _ = next(iter(te_loader))
xb_xai    = xb_xai[:1].to(best_model_device)
baseline  = torch.zeros_like(xb_xai)

ig = IntegratedGradients(model_scalar_100m)
attrs, delta = ig.attribute(
    xb_xai, baselines=baseline,
    n_steps=50, return_convergence_delta=True
)
print(f'IG convergence delta: {delta.item():.4f}  (lower is better)')

attrs_np          = attrs.abs().squeeze(0).cpu().numpy()  # (T, C, H, W)
channel_importance = attrs_np.mean(axis=(0, 2, 3))        # (C,)
spatial_importance  = attrs_np.mean(axis=(0, 1))           # (H, W)

# ─── Plot 1: Channel importance bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(SURF_VARS, channel_importance, color='teal', alpha=0.8)
ax.set_ylabel('|IG attribution|')
ax.set_title('Surface Variable Importance for 100 m Temperature Reconstruction')
ax.set_xticklabels(SURF_VARS, rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'xai_channel_importance.png'), dpi=150)
plt.show()

# ─── Plot 2: Spatial importance map ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(
    spatial_importance[:101, :241],
    origin='lower', aspect='auto',
    extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    cmap='hot_r'
)
plt.colorbar(im, ax=ax, label='|IG attribution|')
ax.set_title('Spatial Attribution Map — 100 m Temperature')
ax.set_xlabel('Longitude');  ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'xai_spatial_map.png'), dpi=150)
plt.show()
print('XAI plots saved.')

## Cell 9 — Save Final Model & Metadata

In [ ]:
import json
import torch
import numpy as np

best_ckpt = torch.load(
    os.path.join(CKPT_DIR, f'{best_run_name}_best.pth'),
    map_location='cpu', weights_only=True)  # FIX: weights_only

cfg = best_ckpt['config']
final_model = OceanEmbedModel(
    in_ch=N_SURF, latent_ch=cfg['latent_ch'],
    h_ch=cfg['h_ch'], out_depths=N_DEPTHS
)
final_model.load_state_dict(best_ckpt['model_state'])
final_model.eval()

save_path = os.path.join(MODEL_DIR, 'oceanembed_final.pt')
torch.save({
    'model_state_dict':  final_model.state_dict(),
    'model_config':      cfg,
    'architecture':      'OceanEmbedModel_UNet_BiConvGRU_CoordAttention',
    'n_surf_channels':   N_SURF,
    'surf_var_names':    SURF_VARS,
    'n_depths':          N_DEPTHS,
    'target_depths_m':   TARGET_DEPTHS,
    'temporal_window':   T_WIN,
    'pad_lat':           PAD_LAT,
    'pad_lon':           PAD_LON,
    'lat_range':         [LAT_MIN, LAT_MAX],
    'lon_range':         [LON_MIN, LON_MAX],
    'train_period':      [TRAIN_START, TRAIN_END],
    'val_period':        [VAL_START,   VAL_END],
    'test_period':       [TEST_START,  TEST_END],
    'scaler_X_mean':     best_ckpt['scaler_X_mean'],
    'scaler_X_scale':    best_ckpt['scaler_X_scale'],
    'scaler_Y_mean':     best_ckpt['scaler_Y_mean'],
    'scaler_Y_scale':    best_ckpt['scaler_Y_scale'],
    'best_val_loss':     float(best_ckpt['val_loss']),
    'best_epoch':        int(best_ckpt['epoch']),
    'best_run_name':     best_run_name,
}, save_path)
print(f'Model saved → {save_path}')

meta = {
    'model':        'OceanEmbedModel',
    'architecture': 'SatelliteEmbeddingEncoder + BiConvGRU + UNet Decoder + CoordAttention',
    'inputs': {'variables': SURF_VARS, 'resolution': '0.25 deg', 'temporal_window_days': T_WIN},
    'output': {'variable': 'thetao_anomaly (°C, relative to daily climatology 2011-2019)', 'depths_m': TARGET_DEPTHS},
    'data_sources': {
        'SST':      'METOFFICE-GLO-SST-L4-REP-OBS-SST (OSTIA)',
        'SSS':      'cmems_obs-mob_glo_phy-sss_my_multi_P1D (SMOS+SMAP)',
        'SSH':      'c3s_obs-sl_glo_phy-ssh_my_twosat-l4-duacs-0.25deg_P1D (DUACS)',
        'Winds':    'cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H (CMEMS)',
        'Currents': 'cmems_obs-mob_glo_phy-cur_my_0.25deg_P1D-m (MULTIOBS)',
        'Target':   'cmems_mod_glo_phy_my_0.083deg_P1D-m (GLORYS12)',
    },
    'training': {
        'train': [TRAIN_START, TRAIN_END], 'val': [VAL_START, VAL_END], 'test': [TEST_START, TEST_END],
        'best_config': cfg, 'best_val_loss': float(best_ckpt['val_loss'])
    },
    'domain': {'lat': [LAT_MIN, LAT_MAX], 'lon': [LON_MIN, LON_MAX]}
}
meta_path = os.path.join(MODEL_DIR, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Metadata saved → {meta_path}')

# Quick inference smoke test
dummy_input = torch.zeros(1, T_WIN, N_SURF, PAD_LAT, PAD_LON)
with torch.no_grad():
    dummy_out = final_model(dummy_input)
print(f'Inference smoke test OK: output shape {tuple(dummy_out.shape)}')
print('\nAll done. Pipeline complete! 🎉')